# Phase 6: Custom Viterbi Algorithm vs hmmlearn Reference

**Objective**: Implement the Viterbi decoding algorithm from scratch and benchmark it against `hmmlearn`'s reference implementation.

### Mathematical Formulation (Log-Domain Dynamic Programming):
1. **Initialization** ($t = 0$):
   $$\delta_0(i) = \log \pi_i + \log b_i(O_0), \quad \psi_0(i) = 0$$
2. **Recursion** ($t = 1, \dots, T-1$):
   $$\delta_t(j) = \max_{i} \left[ \delta_{t-1}(i) + \log A_{ij} \right] + \log b_j(O_t)$$
   $$\psi_t(j) = \arg\max_{i} \left[ \delta_{t-1}(i) + \log A_{ij} \right]$$
3. **Termination**:
   $$\log P^* = \max_{i} \delta_{T-1}(i), \quad q_T^* = \arg\max_{i} \delta_{T-1}(i)$$
4. **Backtracking** ($t = T-2, \dots, 0$):
   $$q_t^* = \psi_{t+1}(q_{t+1}^*)$$

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.custom_viterbi import (
    GaussianHMMParams,
    viterbi_decode,
    viterbi_decode_vectorised,
    compare_with_reference,
)
from src.hmm_model import load_model
from src.sequences import build_sequences_from_features, SequenceNormaliser, split_sequences_chronologically

model, state_map = load_model('../results/model.pkl')
params = GaussianHMMParams.from_hmmlearn(model)
print(f"Loaded HMM with K={params.n_states} states and D={params.n_features} features.")

## 1. Load Test Sequence & Run Custom Viterbi

In [ ]:
feat_df = pd.read_csv('../data/processed/features_all.csv')
sequences, lengths, meta = build_sequences_from_features(feat_df)
(tr, va, test_seqs, tr_m, va_m, test_meta) = split_sequences_chronologically(sequences, meta, 0.7, 0.15, 1)

norm = SequenceNormaliser()
norm.fit(tr)
test_norm = norm.transform(test_seqs)

test_obs = test_norm[0]
print(f"Decoding test sequence of length T={len(test_obs)}...")

custom_states, custom_lp = viterbi_decode_vectorised(params, test_obs)
ref_lp, ref_states = model.decode(test_obs)

agreement = np.mean(custom_states == ref_states)
print(f"Sequence Agreement: {agreement:.2%}")
print(f"Custom Log Prob: {custom_lp:.4f} vs Ref Log Prob: {ref_lp:.4f}")

## 2. Benchmark Across All Test Sequences

In [ ]:
cmp_df = pd.read_csv('../results/model_comparison/viterbi_comparison.csv')
display(cmp_df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(range(len(cmp_df)), cmp_df['custom_time_ms'], width=0.35, label='Custom Viterbi', color='#3498db')
axes[0].bar(np.arange(len(cmp_df)) + 0.35, cmp_df['ref_time_ms'], width=0.35, label='hmmlearn decode()', color='#95a5a6')
axes[0].set_ylabel("Inference Time (ms)")
axes[0].set_title("Latency Comparison", fontweight="bold")
axes[0].legend()

axes[1].bar(range(len(cmp_df)), cmp_df['sequence_agreement']*100, color='#2ecc71', edgecolor='black')
axes[1].set_ylabel("Path Agreement (%)")
axes[1].set_ylim(0, 105)
axes[1].set_title("Custom vs Reference Agreement", fontweight="bold")
plt.tight_layout()
plt.show()